# Invoice Chaser — Data Analysis

**Purpose:** validate the assumptions our agent's logic is built on, using the real IBM Accounts Receivable dataset — not guesses.

This notebook covers:
1. Dataset overview
2. Payment behavior distribution (`DaysLate`)
3. Validating our client risk-score thresholds (`src/client_risk.py`)
4. Disputed vs. non-disputed payment behavior
5. Country-level payment patterns
6. End-to-end sanity check of the pipeline (`src/data_loader.py`, `src/client_risk.py`)

Run this from the **repo root** — it expects `data/accounts_receivable.csv` and imports directly from `src/`.


## 1. Setup & Data Overview

In [ ]:
import sys
sys.path.insert(0, "src")

import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_invoices, get_open_invoices, reminder_tier, DEFAULT_DATA_PATH
from client_risk import compute_client_risk, _risk_level

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (9, 4)

df = load_invoices(str(DEFAULT_DATA_PATH))
print(f"{len(df)} invoices loaded")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


**Sanity checks:**
- No missing values expected across all 12 columns.
- Every invoice should have the same payment term — confirm below.


In [ ]:
term = (df["DueDate"] - df["InvoiceDate"]).dt.days
term.value_counts()


## 2. Payment Behavior — `DaysLate` Distribution

This is the number that drives everything downstream: overdue detection, reminder tone, and risk scoring.


In [ ]:
df["DaysLate"].describe()


In [ ]:
fig, ax = plt.subplots()
df["DaysLate"].plot(kind="hist", bins=40, ax=ax, color="#B3382C", edgecolor="white")
ax.set_title("Distribution of DaysLate across all settled invoices")
ax.set_xlabel("Days late")
plt.show()


In [ ]:
for p in [25, 50, 60, 75, 85, 90, 95, 99]:
    print(f"{p}th percentile: {df['DaysLate'].quantile(p/100):.1f} days")


**Takeaway:** most invoices settle on time or very close to it (median = 0 days late). The distribution has a long right tail — a small share of invoices are dramatically later than the rest. This is exactly the kind of distribution that justifies *tiered* handling instead of one-size-fits-all reminders.


## 3. Validating Our Risk-Score Thresholds

`src/client_risk.py` currently classifies a client's payment history as:

| Tier | Rule |
|---|---|
| Low | avg days late ≤ 2 |
| Medium | avg days late ≤ 10 |
| High | avg days late > 10 |

These started as reasonable-sounding defaults. Let's check where they actually fall on the real distribution, and how many clients land in each tier.


In [ ]:
for cutoff in [2, 10]:
    pct = (df["DaysLate"] <= cutoff).mean() * 100
    print(f"DaysLate <= {cutoff}: {pct:.1f}% of all invoices fall at or under this")


In [ ]:
snapshot_date = df["SettledDate"].max().strftime("%Y-%m-%d")
risk_df = compute_client_risk(df, snapshot_date)

print(f"Snapshot date used: {snapshot_date}")
print(f"Clients scored: {len(risk_df)}\n")
print(risk_df["risk_level"].value_counts())


In [ ]:
fig, ax = plt.subplots()
risk_df["avg_days_late"].plot(kind="hist", bins=30, ax=ax, color="#2F6F68", edgecolor="white")
for cutoff, label in [(2, "Low/Medium cutoff"), (10, "Medium/High cutoff")]:
    ax.axvline(cutoff, color="#B3382C", linestyle="--")
    ax.text(cutoff + 0.3, ax.get_ylim()[1]*0.9, label, rotation=90, fontsize=8)
ax.set_title("Client-level avg_days_late, with current risk thresholds marked")
ax.set_xlabel("Average days late per client")
plt.show()


**Conclusion:** *(fill in after running — see the printed tier breakdown above)*. If the `High` tier ends up capturing roughly the worst-behaving 10–15% of clients rather than, say, half of them, the thresholds are doing their job: flagging genuine outliers instead of over-flagging everyone. If the split looks off (e.g. almost no one lands in `Medium`, or `High` captures 40%+ of clients), that's a sign the cutoffs need adjusting — change the numbers in `_risk_level()` in `src/client_risk.py` and re-run this cell.


## 4. Disputed vs. Non-Disputed Invoices

`Disputed` is a column we're not using anywhere in the agent's logic yet. Does it actually correlate with late payment? If so, our action plan / risk scoring should probably account for it.


In [ ]:
dispute_compare = df.groupby("Disputed")["DaysLate"].agg(["mean", "median", "count"])
dispute_compare


In [ ]:
fig, ax = plt.subplots()
df.boxplot(column="DaysLate", by="Disputed", ax=ax)
ax.set_title("DaysLate by dispute status")
plt.suptitle("")
plt.show()


**Takeaway:** *(fill in after running)* — if disputed invoices are meaningfully later on average, that's a strong case for treating disputed invoices differently in the action plan (e.g. "resolve the dispute" instead of "send a firmer reminder," since a firmer tone on a genuine dispute could backfire).


## 5. Country-Level Payment Patterns


In [ ]:
country_stats = df.groupby("countryCode")["DaysLate"].agg(["mean", "count"]).sort_values("mean", ascending=False)
country_stats


In [ ]:
fig, ax = plt.subplots()
country_stats["mean"].plot(kind="bar", ax=ax, color="#C89A3E")
ax.set_title("Average DaysLate by country code")
ax.set_ylabel("Avg days late")
plt.show()


## 6. End-to-End Pipeline Sanity Check

Trace a handful of real invoices through the full pipeline — overdue detection → tier classification — and eyeball whether the output makes sense.


In [ ]:
check_date = df["DueDate"].quantile(0.6).strftime("%Y-%m-%d")
open_df = get_open_invoices(df, check_date)

print(f"As of {check_date}: {len(open_df)} invoices were open, {open_df['IsOverdue'].sum()} overdue.\n")

sample = open_df[open_df["IsOverdue"]].sort_values("DaysOverdue", ascending=False).head(8).copy()
sample["Tier"] = sample["DaysOverdue"].apply(lambda d: reminder_tier(int(d)))
sample[["customerID", "invoiceNumber", "InvoiceAmount", "DaysOverdue", "Tier"]]


**QA checklist — confirm manually:**
- [ ] Invoices with more `DaysOverdue` get equal-or-firmer tiers than ones with fewer (monotonic).
- [ ] No invoice shows `IsOverdue = True` with a due date in the future relative to the snapshot.
- [ ] Amounts and customer IDs match what's in the raw CSV for a couple of spot-checked rows.

If all three hold, the core pipeline is trustworthy going into the demo.


## 7. Summary for the README / Pitch

Fill these in once the notebook has been run against the real data:

- Total invoices analyzed: **___**
- Standard payment term: **___ days**
- Median days late: **___**
- % of invoices settled on time or early: **___%**
- Risk tier breakdown: Low **___%**, Medium **___%**, High **___%**
- Disputed invoices are on average **___ days later / earlier** than non-disputed ones
- Highest-risk country code: **___**

*Analysis by Eve — data analysis lead, Invoice Chaser (AWS Agents for Humans Hackathon).*
